# Setup and load all models from models/ folder

In [6]:
import pickle
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models"

model_names = ["RW", "DNS", "Ridge", "XGBoost"]

model_preds = {}
model_actuals = {}
model_metrics = {}

for name in model_names:
    path = MODEL_DIR / f"{name}.pkl"
    with open(path, "rb") as f:
        bundle = pickle.load(f)

    print(f"Loaded {name} from {path}, keys: {list(bundle.keys())}")

    model_preds[name]   = bundle["predictions"]
    model_actuals[name] = bundle["actuals"]
    model_metrics[name] = bundle["metrics"]   # <--- NEW

print("Models loaded:", list(model_preds.keys()))


Loaded RW from /Users/chriss/Desktop/Studium/Semester/7. Semester/Bachelorarbeit/yield-curve-forecasting/yield-curve-forecasting/models/RW.pkl, keys: ['predictions', 'actuals', 'metrics']
Loaded DNS from /Users/chriss/Desktop/Studium/Semester/7. Semester/Bachelorarbeit/yield-curve-forecasting/yield-curve-forecasting/models/DNS.pkl, keys: ['predictions', 'actuals', 'metrics']
Loaded Ridge from /Users/chriss/Desktop/Studium/Semester/7. Semester/Bachelorarbeit/yield-curve-forecasting/yield-curve-forecasting/models/Ridge.pkl, keys: ['predictions', 'actuals', 'metrics', 'hyperparameters']
Loaded XGBoost from /Users/chriss/Desktop/Studium/Semester/7. Semester/Bachelorarbeit/yield-curve-forecasting/yield-curve-forecasting/models/XGBoost.pkl, keys: ['predictions', 'actuals', 'metrics', 'hyperparameters']
Models loaded: ['RW', 'DNS', 'Ridge', 'XGBoost']


In [7]:
import pandas as pd
import numpy as np
from pathlib import Path

# ----------------- Paths -----------------
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

# ----------------- Load raw FRED data -----------------
DGS1 = pd.read_csv(DATA_DIR / "DGS1.csv")
DGS2 = pd.read_csv(DATA_DIR / "DGS2.csv")
DGS5 = pd.read_csv(DATA_DIR / "DGS5.csv")
DGS10 = pd.read_csv(DATA_DIR / "DGS10.csv")

# 2y, 5y, 10y panel
merged = (
    DGS2.merge(DGS5, on="observation_date", how="inner")
        .merge(DGS10, on="observation_date", how="inner")
)
merged = merged.dropna(subset=["DGS2", "DGS5", "DGS10"])
merged = merged.set_index("observation_date")
merged.index = pd.to_datetime(merged.index, errors="coerce")

# Short rate (1y) as separate series
DGS1 = DGS1.dropna(subset=["DGS1"])
DGS1 = DGS1.set_index("observation_date")
DGS1.index = pd.to_datetime(DGS1.index, errors="coerce")

short_rate = DGS1["DGS1"]   # <-- THIS is what you pass to econ functions

# ----------------- Settings -----------------
maturity_cols = ["DGS2", "DGS5", "DGS10"]
horizons = [1, 5, 10, 30]
idx = merged.index
train_start = pd.Timestamp("2009-01-02")
train_end   = pd.Timestamp("2018-12-31")

test_start  = pd.Timestamp("2019-01-02")
test_end    = pd.Timestamp("2025-11-25")  


# Robustness Check of RMSE

In [3]:
# =====================================================
# RMSE robustness: half–half split of the OOS period
# Split date chosen to equalize sample sizes (by number of OOS dates)
# =====================================================

def rmse(a, f):
    a = np.asarray(a)
    f = np.asarray(f)
    return np.sqrt(np.mean((a - f) ** 2))

# ---- Build OOS date index and pick split date to equalize counts ----
oos_idx = idx[(idx >= test_start) & (idx <= test_end)].sort_values()
if len(oos_idx) < 10:
    raise ValueError("OOS index seems too short — check idx/test_start/test_end.")

mid_pos = len(oos_idx) // 2
split_date = oos_idx[mid_pos]  # equal-count split point (mid-2022-ish in your sample)
print(f"Equal-count OOS split date: {split_date.date()}  "
      f"(N1={mid_pos}, N2={len(oos_idx)-mid_pos}, N_total={len(oos_idx)})")

# ---- Helper to align and compute RMSE over a mask of target dates ----
def rmse_over_period(y_true, y_pred, date_mask):
    # y_true and y_pred are pd.Series with DateTimeIndex
    df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred}).dropna()
    if df.empty:
        return np.nan, 0
    df = df.loc[df.index.intersection(date_mask)]
    df = df.dropna()
    if df.empty:
        return np.nan, 0
    return rmse(df["y_true"], df["y_pred"]), len(df)

# ---- Compute RMSEs ----
rows = []

# Define the three evaluation date sets (based on the target date t+h)
dates_full = oos_idx
dates_1 = oos_idx[oos_idx < split_date]
dates_2 = oos_idx[oos_idx >= split_date]

for maturity in maturity_cols:
    for h in horizons:
        key = (maturity, h)

        # actuals are the realized y_{t+h} aligned to the forecast evaluation date (your bundle design)
        # We use RW actuals as canonical since they should be identical across models
        if key not in model_actuals["RW"]:
            continue
        y_true_all = model_actuals["RW"][key]

        for model in model_names:
            if key not in model_preds[model]:
                continue
            y_pred_all = model_preds[model][key]

            # Align to OOS and compute period RMSEs
            rmse_full, n_full = rmse_over_period(y_true_all, y_pred_all, dates_full)
            rmse_1, n_1 = rmse_over_period(y_true_all, y_pred_all, dates_1)
            rmse_2, n_2 = rmse_over_period(y_true_all, y_pred_all, dates_2)

            rows.append({
                "Maturity": maturity,
                "Horizon": h,
                "Model": model,
                "RMSE_Full": rmse_full,
                "RMSE_2019_to_split": rmse_1,
                "RMSE_split_to_2025": rmse_2,
                "N_Full": n_full,
                "N_2019_to_split": n_1,
                "N_split_to_2025": n_2,
            })

rmse_robustness_df = pd.DataFrame(rows).sort_values(["Horizon", "Maturity", "Model"])

# Round for display
for c in ["RMSE_Full", "RMSE_2019_to_split", "RMSE_split_to_2025"]:
    rmse_robustness_df[c] = rmse_robustness_df[c].round(6)

display(rmse_robustness_df)

# Optional: quick check that sample sizes are near-equal for most rows
print("Unique (N_2019_to_split, N_split_to_2025) pairs:",
      sorted(set(zip(rmse_robustness_df["N_2019_to_split"], rmse_robustness_df["N_split_to_2025"])))[:10])
rmse_robustness_df.to_csv("rmse_robustness_df.csv", index=False)



Equal-count OOS split date: 2022-06-13  (N1=863, N2=863, N_total=1726)


,Maturity,Horizon,Model,RMSE_Full,RMSE_2019_to_split,RMSE_split_to_2025,N_Full,N_2019_to_split,N_split_to_2025
33,DGS10,1,DNS,0.059081,0.050736,0.066377,1725,862,863
32,DGS10,1,RW,0.058991,0.050602,0.066327,1726,863,863
34,DGS10,1,Ridge,0.059137,0.050871,0.066373,1725,862,863
35,DGS10,1,XGBoost,0.061816,0.052475,0.070028,1712,862,850
1,DGS2,1,DNS,0.061312,0.042503,0.075561,1725,862,863
0,DGS2,1,RW,0.061234,0.042448,0.075481,1726,863,863
2,DGS2,1,Ridge,0.061421,0.042505,0.075737,1725,862,863
3,DGS2,1,XGBoost,0.065850,0.043788,0.082396,1712,862,850
17,DGS5,1,DNS,0.062685,0.048506,0.074190,1725,862,863
16,DGS5,1,RW,0.062582,0.048337,0.074138,1726,863,863


Unique (N_2019_to_split, N_split_to_2025) pairs: [(833, 850), (833, 863), (853, 850), (853, 863), (858, 850), (858, 863), (862, 850), (862, 863), (863, 863)]


In [4]:
maturity = "DGS10"
h = 10
key = (maturity, h)

for model in model_names:
    y_pred = model_preds[model][key]
    y_true = model_actuals["RW"][key]

    overlap = y_pred.index.intersection(y_true.index)
    print(model, "pred idx min/max:", y_pred.index.min(), y_pred.index.max(),
          "| true idx min/max:", y_true.index.min(), y_true.index.max(),
          "| overlap:", len(overlap),
          "| pred-only:", len(y_pred.index.difference(y_true.index)),
          "| true-only:", len(y_true.index.difference(y_pred.index)))


RW pred idx min/max: 2019-01-02 00:00:00 2025-11-25 00:00:00 | true idx min/max: 2019-01-02 00:00:00 2025-11-25 00:00:00 | overlap: 1726 | pred-only: 0 | true-only: 0
DNS pred idx min/max: 2019-01-16 00:00:00 2025-11-25 00:00:00 | true idx min/max: 2019-01-02 00:00:00 2025-11-25 00:00:00 | overlap: 1716 | pred-only: 0 | true-only: 10
Ridge pred idx min/max: 2019-01-16 00:00:00 2025-11-25 00:00:00 | true idx min/max: 2019-01-02 00:00:00 2025-11-25 00:00:00 | overlap: 1716 | pred-only: 0 | true-only: 10
XGBoost pred idx min/max: 2019-01-16 00:00:00 2025-11-05 00:00:00 | true idx min/max: 2019-01-02 00:00:00 2025-11-25 00:00:00 | overlap: 1703 | pred-only: 0 | true-only: 23


# Diebold-Marino Test

In [8]:
import numpy as np
from scipy.stats import norm

def dm_test(errors_model1, errors_model2, h=1, lag=None, apply_hln=True):
    """
    Diebold–Mariano (DM) test for equal predictive accuracy using squared-error loss,
    with Newey–West HAC variance and optional Harvey–Leybourne–Newbold (HLN) small-sample
    correction.

    Parameters
    ----------
    errors_model1 : array-like
        Forecast errors of model 1 (y - y_hat1), length N.
    errors_model2 : array-like
        Forecast errors of model 2 (y - y_hat2), length N.
    h : int, optional
        Forecast horizon (in periods). Used for HLN correction and (if lag is None)
        for setting the Newey–West truncation lag. Default is 1.
    lag : int, optional
        Newey–West truncation lag for HAC variance of the loss differential.
        If None, lag is set to max(h-1, 0).
    apply_hln : bool, optional
        If True, apply the HLN small-sample correction (Harvey et al., 1997) to the
        DM statistic. Default is True.

    Returns
    -------
    dm_stat : float
        DM statistic (HLN-adjusted if apply_hln=True).
    p_value : float
        Two-sided p-value under asymptotic N(0,1).
    """

    e1 = np.asarray(errors_model1, dtype=float)
    e2 = np.asarray(errors_model2, dtype=float)

    # Drop NaNs in parallel
    mask = np.isfinite(e1) & np.isfinite(e2)
    e1 = e1[mask]
    e2 = e2[mask]

    if e1.shape != e2.shape:
        raise ValueError("Error series must have the same length after NaN removal.")

    N = len(e1)
    if N < 5:
        raise ValueError("Not enough observations for DM test.")

    if h < 1:
        raise ValueError("h must be >= 1.")

    # Squared-error loss and loss differential
    d = (e1 ** 2) - (e2 ** 2)
    d_bar = np.mean(d)

    # Newey–West HAC variance of d_t
    if lag is None:
        lag = max(h - 1, 0)
    if lag < 0:
        raise ValueError("lag must be >= 0.")

    d_centered = d - d_bar
    gamma0 = np.dot(d_centered, d_centered) / N
    var_d = gamma0

    for k in range(1, lag + 1):
        cov = np.dot(d_centered[k:], d_centered[:-k]) / N
        weight = 1.0 - k / (lag + 1)  # Bartlett weight
        var_d += 2.0 * weight * cov

    if var_d <= 0 or not np.isfinite(var_d):
        raise ValueError(f"Non-positive/invalid HAC variance estimate: var_d={var_d}")

    dm_stat = d_bar / np.sqrt(var_d / N)

    # Harvey–Leybourne–Newbold (1997) small-sample correction
    if apply_hln:
        # HLN factor: sqrt((N + 1 - 2h + h(h/N)) / N)
        hln_factor = np.sqrt((N + 1 - 2 * h + (h * (h / N))) / N)
        dm_stat = dm_stat * hln_factor

    # Two-sided p-value under asymptotic N(0,1)
    p_value = 2 * (1 - norm.cdf(np.abs(dm_stat)))

    return dm_stat, p_value


In [9]:
maturity_cols = ["DGS2", "DGS5", "DGS10"]
horizons = [1, 5, 10, 30]

dm_results = []

benchmark_name = "RW"

if benchmark_name not in model_actuals:
    raise ValueError(f"Benchmark model '{benchmark_name}' not loaded.")

# Use RW's actual series as canonical ground truth
rw_actual_series = model_actuals[benchmark_name]

for maturity in maturity_cols:
    for h in horizons:
        key = (maturity, h)

        # Use RW's actual series as canonical ground truth for this (maturity, h)
        if key not in rw_actual_series:
            continue

        actual = rw_actual_series[key].copy()
        actual.name = "actual"

        # Build a dict of aligned prediction series for all models that exist
        preds_for_key = {}
        for model_name, pred_dict in model_preds.items():
            if key not in pred_dict:
                continue
            preds_for_key[model_name] = pred_dict[key].rename(model_name)

        # Need at least RW + one other model with forecasts
        if benchmark_name not in preds_for_key or len(preds_for_key) < 2:
            continue

        # Compare each non-RW model against RW
        for model_name, pred_series in preds_for_key.items():
            if model_name == benchmark_name:
                continue

            # Pairwise alignment: actual, RW, and the other model
            df_pair = pd.concat(
                [
                    actual,
                    preds_for_key[benchmark_name],  # RW predictions
                    pred_series                     # other model predictions
                ],
                axis=1,
                join="inner"
            ).dropna()

            if df_pair.empty:
                continue

            # Forecast errors: e = actual - prediction
            err_rw    = df_pair["actual"] - df_pair[benchmark_name]
            err_other = df_pair["actual"] - df_pair[model_name]

            # Diebold–Mariano test (RW as Model 1, other model as Model 2)
            dm_stat, p_val = dm_test(err_rw.values, err_other.values, h=h)

            dm_results.append({
                "Maturity": maturity,
                "Horizon":  h,
                "Model_1":  benchmark_name,
                "Model_2":  model_name,
                "DM_stat":  dm_stat,
                "p_value":  p_val,
                "N":        len(df_pair)
            })

dm_results_df = pd.DataFrame(dm_results)
dm_results_df = dm_results_df.sort_values(["Maturity", "Horizon", "Model_2"])

display(dm_results_df)

,Maturity,Horizon,Model_1,Model_2,DM_stat,p_value,N
24,DGS10,1,RW,DNS,-2.075474,0.037943,1725
25,DGS10,1,RW,Ridge,-1.350084,0.176989,1725
26,DGS10,1,RW,XGBoost,-3.804639,0.000142,1712
27,DGS10,5,RW,DNS,-2.535540,0.011227,1721
28,DGS10,5,RW,Ridge,-1.543898,0.122613,1721
29,DGS10,5,RW,XGBoost,-3.966589,0.000073,1708
30,DGS10,10,RW,DNS,-2.591153,0.009565,1716
31,DGS10,10,RW,Ridge,-2.752886,0.005907,1716
32,DGS10,10,RW,XGBoost,-2.617852,0.008849,1703
33,DGS10,30,RW,DNS,-2.211876,0.026975,1696


In [11]:
maturity_cols = ["DGS2", "DGS5", "DGS10"]
horizons      = [1, 5, 10, 30]

dm_results = []

# -------------------------------------------------
# Benchmark: DNS
# -------------------------------------------------
benchmark_name = "DNS"
models_to_compare = ["Ridge", "XGBoost"]   # you can add "RW" here if you change your mind

if benchmark_name not in model_actuals:
    raise ValueError(f"Benchmark model '{benchmark_name}' not loaded.")

# Use DNS' actual series as canonical ground truth (should be identical across models)
dns_actual_series = model_actuals[benchmark_name]

for maturity in maturity_cols:
    for h in horizons:
        key = (maturity, h)

        # Check that we have actuals for this maturity/horizon
        if key not in dns_actual_series:
            continue

        # Actual LEVEL yields at forecast dates
        actual = dns_actual_series[key].copy()
        actual.name = "actual"

        # -------------------------------------------------
        # Collect prediction series for this (maturity, h)
        # -------------------------------------------------
        preds_for_key = {}
        for model_name, pred_dict in model_preds.items():
            if key not in pred_dict:
                continue
            preds_for_key[model_name] = pred_dict[key].rename(model_name)

        # Need benchmark + at least one comparison model
        if benchmark_name not in preds_for_key:
            continue

        # -------------------------------------------------
        # Compare each model in models_to_compare vs DNS
        # -------------------------------------------------
        for model_name in models_to_compare:
            if model_name not in preds_for_key:
                continue
            if model_name == benchmark_name:
                continue

            pred_bench = preds_for_key[benchmark_name]
            pred_other = preds_for_key[model_name]

            # Pairwise alignment: actual, DNS, and the other model
            df_pair = pd.concat(
                [actual, pred_bench, pred_other],
                axis=1,
                join="inner"
            ).dropna()

            if df_pair.empty:
                continue

            # Forecast errors: e = actual - prediction
            err_bench = df_pair["actual"] - df_pair[benchmark_name]   # DNS errors
            err_other = df_pair["actual"] - df_pair[model_name]       # other model

            # Diebold–Mariano test (DNS as Model 1, other model as Model 2)
            dm_stat, p_val = dm_test(err_bench.values, err_other.values, h=h)

            dm_results.append({
                "Maturity": maturity,
                "Horizon":  h,
                "Model_1":  benchmark_name,
                "Model_2":  model_name,
                "DM_stat":  dm_stat,
                "p_value":  p_val,
                "N":        len(df_pair),
            })

# -------------------------------------------------
# Build results DataFrame
# -------------------------------------------------
dm_dns_benchmark_df = (
    pd.DataFrame(dm_results)
    .sort_values(["Maturity", "Horizon", "Model_2"])
    .reset_index(drop=True)
)

display(dm_dns_benchmark_df)


,Maturity,Horizon,Model_1,Model_2,DM_stat,p_value,N
0,DGS10,1,DNS,Ridge,-0.624639,0.532208,1725
1,DGS10,1,DNS,XGBoost,-3.686500,0.000227,1712
2,DGS10,5,DNS,Ridge,0.764102,0.444806,1721
3,DGS10,5,DNS,XGBoost,-3.672787,0.000240,1708
4,DGS10,10,DNS,Ridge,-0.366486,0.714003,1716
5,DGS10,10,DNS,XGBoost,-2.172855,0.029791,1703
6,DGS10,30,DNS,Ridge,-2.741397,0.006118,1696
7,DGS10,30,DNS,XGBoost,-2.883782,0.003929,1683
8,DGS2,1,DNS,Ridge,-0.937815,0.348339,1725
9,DGS2,1,DNS,XGBoost,-3.879112,0.000105,1712


# Root-Mean-Squared Error

In [11]:
# ----------------------------------------------------
# RMSE comparison table across models and horizons
# ----------------------------------------------------

rmse_tables = []

for model_name, df in model_metrics.items():
    if "RMSE" not in df.columns:
        print(f"⚠️ No RMSE column found for {model_name}, skipping.")
        continue

    rmse_tables.append(
        df[["RMSE"]].rename(columns={"RMSE": model_name})
    )

rmse_compare_df = pd.concat(rmse_tables, axis=1).sort_index()

display(rmse_compare_df)


RW       DNS     Ridge   XGBoost
Maturity Horizon                                        
DGS10    1        0.059004  0.059081  0.059137  0.061816
         5        0.126819  0.127669  0.127458  0.136784
         10       0.179402  0.181670  0.181782  0.193798
         30       0.328191  0.338187  0.346037  0.388307
DGS2     1        0.061250  0.061312  0.061421  0.065850
         5        0.131312  0.131994  0.132197  0.144119
         10       0.188868  0.190579  0.190964  0.210959
         30       0.359041  0.366263  0.369247  0.413400
DGS5     1        0.062598  0.062685  0.062806  0.065312
         5        0.134188  0.135193  0.135146  0.141190
         10       0.190236  0.192971  0.193134  0.206624
         30       0.349621  0.361938  0.364119  0.414612